In [1]:
import pandas as pd

In [2]:
def parse_vcf(input_filepath: str) -> list:
    variants_list = []
    with open(input_filepath, 'r') as input_file:
        for row in input_file:
            row = row.strip('\n')
            if row.startswith('##'):
                continue
            elif row.startswith('#'):
                header_columns = row.strip('#').split('\t')
            else:
                splitted_row = row.split('\t')
                var_dict = dict(zip(header_columns, splitted_row))
                # Format - Sample
                format_ = var_dict['FORMAT'].split(':')
                sample = var_dict[header_columns[-1]].split(':')
                sample_format_data = dict(zip(format_, sample))
                # Add data
                var_dict |= sample_format_data
                variants_list.append(var_dict)
    
    return variants_list

In [3]:
roommate_95_df = pd.DataFrame(parse_vcf('../vcf/VarScan_results_95.vcf'))
roommate_01_df = pd.DataFrame(parse_vcf('../vcf/VarScan_results_0.1.vcf'))
control_1_01_df = pd.DataFrame(parse_vcf('../vcf/SRR1705858_0.1.vcf'))
control_2_01_df = pd.DataFrame(parse_vcf('../vcf/SRR1705859_0.1.vcf'))
control_3_01_df = pd.DataFrame(parse_vcf('../vcf/SRR1705860_0.1.vcf'))

In [4]:
roommate_95_df = roommate_95_df.assign(source='roommate_95')
roommate_01_df = roommate_01_df.assign(source='roommate_01')
control_1_01_df = control_1_01_df.assign(source='control_1_01')
control_2_01_df = control_2_01_df.assign(source='control_2_01')
control_3_01_df = control_3_01_df.assign(source='control_3_01')

In [5]:
combined_df = pd.concat([roommate_95_df, roommate_01_df, control_1_01_df, control_2_01_df, control_3_01_df], ignore_index=True)

In [6]:
combined_df_short = combined_df[['POS', 'REF', 'ALT', 'FREQ', 'source']]
combined_df_short['FREQ'] = combined_df_short['FREQ'].str.replace('%', '').str.replace(',', '.')
combined_df_short['FREQ'] = combined_df_short['FREQ'].astype('float')

/var/folders/rf/k3pdmpqx0bjfvwyn2q6sw5v00000gn/T/ipykernel_99717/1117335534.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_df_short['FREQ'] = combined_df_short['FREQ'].str.replace('%', '').str.replace(',', '.')
/var/folders/rf/k3pdmpqx0bjfvwyn2q6sw5v00000gn/T/ipykernel_99717/1117335534.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_df_short['FREQ'] = combined_df_short['FREQ'].astype('float')


In [7]:
combined_df_short

,POS,REF,ALT,FREQ,source
0,72,A,G,99.96,roommate_95
1,117,C,T,99.82,roommate_95
2,774,T,C,99.96,roommate_95
3,999,C,T,99.86,roommate_95
4,1260,A,C,99.94,roommate_95
...,...,...,...,...,...
191,1421,A,G,0.37,control_3_01
192,1460,A,G,0.26,control_3_01
193,1482,A,G,0.23,control_3_01
194,1580,T,C,0.27,control_3_01


In [8]:
freq_stats = combined_df_short.groupby('source')['FREQ'].agg(['mean', 'std'])
freq_stats

,mean,std
source,,
control_1_01,0.256491,0.071726
control_2_01,0.236923,0.052376
control_3_01,0.250328,0.078038
roommate_01,23.999048,43.482824
roommate_95,99.908000,0.064187


In [9]:
freq_stats_controls = freq_stats.head(3)
freq_stats_controls

,mean,std
source,,
control_1_01,0.256491,0.071726
control_2_01,0.236923,0.052376
control_3_01,0.250328,0.078038


In [ ]:
freq_trashold = freq_stats_controls['mean'].mean() + (freq_stats_controls['std'].mean() * 3)
freq_trashold

np.float64(0.4500541648741355)

In [ ]:
selected_variants_df = combined_df_short.query('source == "roommate_01" and FREQ > @freq_trashold')
selected_variants_df[selected_variants_df.columns[:-1]].to_csv('../roommate_0.001_variants.tsv', sep='\t', index=False)
selected_variants_df

,POS,REF,ALT,FREQ,source
5,72,A,G,99.96,roommate_01
6,117,C,T,99.82,roommate_01
9,307,C,T,0.94,roommate_01
15,774,T,C,99.96,roommate_01
19,999,C,T,99.86,roommate_01
23,1260,A,C,99.94,roommate_01
25,1458,T,C,0.84,roommate_01
